## xLH-ai-bp-vs-hf
Eine exemplarische Umsetzung für die Integration von LLM-Modellen für die Berufspädagogik

In [2]:
import os
import pathlib
import sys
__cwd__ = str(pathlib.Path(os.getcwd())).replace('\\\\', '/')
sys.path.append(__cwd__)
print(__cwd__)


D:\Python\xLH-mims\notebooks\src\ai_bp_hf


In [3]:
import pandas as pd
from dataclasses import dataclass
from pydantic_ai import Tool
from pathlib import Path
from rich import print
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from tools import read_hk
from llm_model import get_model, LlmModel
from object_to_file import base_model_to_file
from config import config
import nest_asyncio
nest_asyncio.apply()

D:\Python\xLH-mims-data\config\xlh_mims_python.env


In [4]:
# OpenAi API Key (Credentials hinterlegt in der Datei ..\xLH-mims-data\config\xlh_mims_python.env)
print(config.openai_api_key[:20])
# falls keine Datei im lokalen Speicher vorhanden ist, wir die Datei env.py

sk-proj-hevDtsISJBCJ

In [5]:
# df = pd.read_pickle("RS_LK.pkl")
# df.head()

In [6]:
@dataclass
class Deps:
    name: str

In [38]:
class RowHkLk(BaseModel):
    id_weiterbildung: str = Field(description='ID der Weiterbildung')
    id: str = Field(description='ID der einzelnen Handlungskompetenz oder Leistungskriterium')
    name: str = Field(description='Name der einzelnen Handlungskompetenz oder Leistungskriterium')

In [39]:
class RowBegruendung(BaseModel):
    hk: list[RowHkLk] = Field(description='Auflistung der einzelnen Handlungskompetenzen')
    lk: list[RowHkLk] = Field(description='Auflistung der einzelnen Leistungskriterien')
    begruendungstext: str = Field(description='Dieser Begründungstext soll sich auf die aufgeführten Handlungskompetenzen und Leistungskriterien beziehen')


In [40]:
class RowTexte(BaseModel):
    titel: str = Field(description='Kurzbeschreibung')
    beschreibung: str = Field(description='ausführliche Beschreibung')
    begruendung: list[RowBegruendung] = Field(description='Die einzelnen Begründungen aufgelistet')

In [41]:
class ResponseCompare(BaseModel):
    text_gemeinsamkeiten: list[RowTexte] = Field(description='Was sind die Gemeinsamkeiten, aufgelistet')
    text_unterschiede: list[RowTexte] = Field(description='Was sind die Unterschiede, aufgelistet')

In [42]:
model = get_model(llm_model=LlmModel.OPENAI_GPT_5_2)  # ToDo: Integration OLLAMA...
agent= Agent(
    model=model,
    system_prompt=('Du analysierst Daten zu den Aus- und Weiterbildungen der schweizer Berufsbildung.'),
    deps_type=Deps,
    tools=[
        # Tool(read_hk, takes_ctx=True),
    ],
    retries=3,
    output_type=ResponseCompare,
)
deps = Deps(name='-')

In [43]:
ts_hk = Path("TS_HK.md").read_text(encoding="utf-8")
ts_lk = Path("TS_LK.md").read_text(encoding="utf-8")
rs_hk = Path("RS_HK.md").read_text(encoding="utf-8")
rs_lk = Path("RS_LK.md").read_text(encoding="utf-8")

response = agent.run_sync(user_prompt=f"Wie unterscheiden sich die beiden Weiterbildungen Rettungssanitäter (RS) und Transportsanitäter (TS)? "
                                      f"Hier sind die entsprechenden Daten dazu:"
                                      f"- hier die Handlungskompetenzen des RS: {rs_hk}"
                                      f"- hier die Leistungskriterien des RS in Verbindung zu den Handlungskompetenzen und Zuteilung des Niveaus: {rs_lk}"
                                      f"- hier die Handlungskompetenzen des TS: {ts_hk}"
                                      f"- hier die Leistungskriterien des TS in Verbindung zu den Handlungskompetenzen und Zuteilung des Niveaus: {ts_lk}"
                          )

In [44]:
result: Quantity = response.output
base_model_to_file(result)
# siehe recipe.json
print(f'Antwort: {result.model_dump_json(indent=4)[:]}')

Antwort: {
    "text_gemeinsamkeiten": [
        {
            "titel": "Gemeinsamer Kernauftrag: präklinische Versorgung, Transport und Patientensicherheit",
            "beschreibung": "Beide Weiterbildungen zielen auf eine sichere präklinische Betreuung und den 
fachgerechten Transport von Patientinnen und Patienten ab – inklusive strukturierter Lage-/Patientenbeurteilung, 
Überwachung, Kommunikation, Dokumentation sowie sicherem Führen von Einsatzfahrzeugen.",
            "begruendung": [
                {
                    "hk": [
                        {
                            "id_weiterbildung": "RS",
                            "id": "rs_hk_03_01",
                            "name": "Patientinnen/Patienten beurteilen und Behandlungsprioritäten festlegen"
                        },
                        {
                            "id_weiterbildung": "RS",
                            "id": "rs_hk_03_02",
                            "name": "Medizinische Sofortmassnahmen durchführen"
                        },
                        {
                            "id_weiterbildung": "RS",
                            "id": "rs_hk_03_05",
                            "name": "Die Überwachung der Patientinnen/Patienten sicherstellen"
                        },
                        {
                            "id_weiterbildung": "RS",
                            "id": "rs_hk_02_01",
                            "name": "Im Team und mit anderen Fachkräften und Berufsgruppen kommunizieren"
                        },
                        {
                            "id_weiterbildung": "RS",
                            "id": "rs_hk_01_05",
                            "name": "Einsatzdokumentation erstellen"
                        },
                        {
                            "id_weiterbildung": "RS",
                            "id": "rs_hk_04_03",
                            "name": "Einsatzfahrzeuge unter allen Gegebenheiten sicher und angepasst führen"
                        },
                        {
                            "id_weiterbildung": "TS",
                            "id": "ts_hk_02_02",
                            "name": "Strukturierte Patientenbeurteilung durchführen und interpretieren"
                        },
                        {
                            "id_weiterbildung": "TS",
                            "id": "ts_hk_02_03",
                            "name": "Lebensrettenden Sofortmassnahmen einleiten"
                        },
                        {
                            "id_weiterbildung": "TS",
                            "id": "ts_hk_02_06",
                            "name": "Patientinnen und Patienten kontinuierlich überwachen und 
Zustandsverschlechterungen frühzeitig erkennen"
                        },
                        {
                            "id_weiterbildung": "TS",
                            "id": "ts_hk_01_04",
                            "name": "Im Team, mit anderen Fachpersonen und Partnerorganisationen kommunizieren und 
kooperieren"
                        },
                        {
                            "id_weiterbildung": "TS",
                            "id": "ts_hk_01_05",
                            "name": "Dokumentation über den Einsatz führen"
                        },
                        {
                            "id_weiterbildung": "TS",
                            "id": "ts_hk_03_03",
                            "name": "Einsatzfahrzeuge unter allen Gegebenheiten sicher und angepasst führen"
                        }
                    ],
                    "lk": [
                        {
                            "id_weiterbildung": "RS",
                            "id": "ts_lk_03_01_092",
                            "name": "Gesundheitszustand beurteilen"
                        },
                        {
                            "id_weiterbildung": "RS",
                

In [45]:
usage = response.usage()
print(f'Input Tokens: {usage.input_tokens} ({usage.input_tokens*1.75*1e-6:0.04f} $)')
print(f'Output Tokens: {usage.output_tokens} ({usage.output_tokens*14.0*1e-6:0.04f} $)')
print(f'Total Tokens: {usage.total_tokens} (total cost: {(usage.input_tokens*1.75*1e-6 + usage.input_tokens*1.75*1e-6):0.04f} $)')

Input Tokens: 33368 (0.0584 $)

Output Tokens: 3241 (0.0454 $)

Total Tokens: 36609 (total cost: 0.1168 $)

In [12]:
settings = agent.model.settings
print(f'Settings: {settings}')
print(f'temperature: {settings.get('temperature')}')
print(f'presence_penalty: {settings.get('presence_penalty')}')
print(f'frequency_penalty: {settings.get('frequency_penalty')}')

Settings: {'temperature': 0.0, 'presence_penalty': 0.0, 'frequency_penalty': 0.0, 'parallel_tool_calls': True, 
'openai_reasoning_effort': 'none', 'openai_reasoning_summary': 'auto', 'openai_text_verbosity': 'low'}

temperature: 0.0

presence_penalty: 0.0

frequency_penalty: 0.0